# YOLO11 + CLIP — Fire Detection & Caption Ranking

**Pipeline:**
1. **YOLO11** (`leeyunjai/yolo11-firedetect`) detects fire/smoke in every image.
2. Images where fire **is** detected are passed to **OpenAI CLIP** (`ViT-B/32`).
3. CLIP compares each fire image against **10 captions** and returns a ranked list — the top caption is the closest semantic match.

### Before running
1. Upload your image folder to **Google Drive → My Drive**.
2. Set `DRIVE_IMAGE_FOLDER` in Cell 1 to the exact folder name.
3. **Allow popups** for `colab.research.google.com`.
4. Recommended runtime: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Cell 1: Mount Google Drive and set image folder ───────────────────────────
from google.colab import drive
import os, zipfile

drive.mount('/content/drive', force_remount=True)

# ↓ Change this to the name of the folder you uploaded to My Drive
DRIVE_IMAGE_FOLDER = 'yolo11images'

DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_IMAGE_FOLDER}'

assert os.path.isdir(DRIVE_DIR), (
    f'Folder not found: {DRIVE_DIR}\n'
    f'Make sure "{DRIVE_IMAGE_FOLDER}" exists inside My Drive on Google Drive.'
)

# Extract ZIPs if present
zip_files = [f for f in os.listdir(DRIVE_DIR) if f.lower().endswith('.zip')]
if zip_files:
    EXTRACT_DIR = f'/content/{DRIVE_IMAGE_FOLDER}_extracted'
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    for zf in zip_files:
        print(f'Extracting {zf} ...')
        with zipfile.ZipFile(os.path.join(DRIVE_DIR, zf), 'r') as z:
            z.extractall(EXTRACT_DIR)
    IMAGE_DIR = EXTRACT_DIR
    print(f'Extracted to {IMAGE_DIR}')
else:
    IMAGE_DIR = DRIVE_DIR
    print(f'Reading images directly from {IMAGE_DIR}')

total = sum(1 for _, _, fs in os.walk(IMAGE_DIR) for f in fs)
print(f'Found {total} file(s) in total')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q ultralytics
!pip install -q git+https://github.com/openai/CLIP.git
print('Done.')

In [ ]:
# ── Cell 3: Load YOLO11 fire-detection model ───────────────────────────────────
import torch
from ultralytics import YOLO

WEIGHTS_PATH = '/content/fire_model.pt'
WEIGHTS_URL  = 'https://huggingface.co/leeyunjai/yolo11-firedetect/resolve/main/firedetect-11s.pt'

if not os.path.exists(WEIGHTS_PATH):
    print('Downloading YOLO11 weights ...')
    os.system(f'wget -q -O {WEIGHTS_PATH} {WEIGHTS_URL}')
else:
    print('Weights already downloaded.')

yolo_model = YOLO(WEIGHTS_PATH)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'YOLO11 loaded  |  device: {DEVICE.upper()}')
print(f'Classes: {yolo_model.names}')

In [ ]:
# ── Cell 4: Configuration ─────────────────────────────────────────────────────
CONF_THRESHOLD = 0.25
IMAGE_EXTS     = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

# 10 captions for CLIP to rank against each fire-detected image
FIRE_CAPTIONS = [
    "a massive forest fire with heavy smoke",
    "a small controlled campfire in a pit",
    "an indoor building fire with visible flames",
    "a car engine on fire on the highway",
    "bright red sunset clouds in the sky",
    "electrical sparks coming from a wire",
    "thick black smoke rising from a factory",
    "a person holding a small lighter flame",
    "firefighters spraying water on a house fire",
    "Fireworks explosion in the night sky",
]

In [ ]:
# ── Cell 5: Run YOLO11 — collect fire-detected image paths ────────────────────
import time
from pathlib import Path

image_paths = sorted(
    p for p in Path(IMAGE_DIR).rglob('*')
    if p.suffix.lower() in IMAGE_EXTS
)
print(f'Found {len(image_paths)} image(s). Running YOLO11 detection ...')

fire_image_paths = []
yolo_results_map = {}   # path → YOLO result (reused for visualisation)
t0 = time.time()

for i, img_path in enumerate(image_paths, 1):
    results = yolo_model.predict(
        source=str(img_path),
        conf=CONF_THRESHOLD,
        device=DEVICE,
        verbose=False,
        save=False,
    )
    result = results[0]
    boxes  = result.boxes

    if boxes is not None and len(boxes) > 0:
        fire_image_paths.append(img_path)
        yolo_results_map[img_path] = result

    if i % 50 == 0 or i == len(image_paths):
        print(f'  [{i}/{len(image_paths)}]  fire so far: {len(fire_image_paths)}  |  elapsed: {time.time()-t0:.1f}s')

print(f'\nYOLO11 done in {time.time()-t0:.1f}s')
print(f'Fire detected in {len(fire_image_paths)}/{len(image_paths)} images')

In [ ]:
# ── Cell 6: Load CLIP model ────────────────────────────────────────────────────
import clip
from PIL import Image

clip_model, preprocess = clip.load('ViT-B/32', device=DEVICE)
print(f'CLIP (ViT-B/32) loaded  |  device: {DEVICE.upper()}')

# Pre-tokenise captions once (shared across all images)
text_tokens = clip.tokenize(FIRE_CAPTIONS).to(DEVICE)

In [ ]:
# ── Cell 7: CLIP caption ranking for every fire-detected image ────────────────
import numpy as np

def rank_captions(img_path):
    """Return sorted list of (caption, score) for img_path."""
    image = preprocess(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits_per_image, _ = clip_model(image, text_tokens)
        probs = logits_per_image.softmax(dim=-1).cpu().numpy()[0]
    return sorted(zip(FIRE_CAPTIONS, probs), key=lambda x: x[1], reverse=True)

clip_results = {}   # path → ranked [(caption, score), ...]

if not fire_image_paths:
    print('No fire-detected images to process with CLIP.')
else:
    print(f'Running CLIP on {len(fire_image_paths)} fire-detected image(s) ...\n')
    for img_path in fire_image_paths:
        ranked = rank_captions(img_path)
        clip_results[img_path] = ranked

        best_caption, best_score = ranked[0]
        print(f'Image : {img_path.name}')
        print(f'  Best match  : "{best_caption}" (score={best_score:.4f})')
        for rank, (cap, score) in enumerate(ranked, 1):
            bar = '█' * int(score * 40)
            print(f'  {rank:2}. {cap:<45} {score:.4f}  {bar}')
        print()

In [ ]:
# ── Cell 8: Visualise — YOLO bounding boxes + best CLIP caption ───────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image as PILImage

N_SHOW = 6   # max images to display

show_paths = fire_image_paths[:N_SHOW]

if not show_paths:
    print('No fire-detected images to visualise.')
else:
    cols = min(len(show_paths), 3)
    rows = (len(show_paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
    axes = np.array(axes).flatten()

    for ax, img_path in zip(axes, show_paths):
        # YOLO annotated frame (BGR → RGB)
        annotated = yolo_results_map[img_path].plot()[:, :, ::-1]
        ax.imshow(annotated)

        best_caption, best_score = clip_results[img_path][0]
        ax.set_title(
            f'{img_path.name}\n"{best_caption}"\n(CLIP score: {best_score:.3f})',
            fontsize=8,
            wrap=True,
        )
        ax.axis('off')

    # Hide unused subplot axes
    for ax in axes[len(show_paths):]:
        ax.set_visible(False)

    plt.suptitle('YOLO11 Fire Detection  +  CLIP Caption Ranking', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 9: Save summary CSV to Google Drive ───────────────────────────────────
import csv

OUTPUT_CSV = '/content/drive/MyDrive/yolo_clip_results.csv'

fieldnames = ['filename', 'best_caption', 'best_score'] + [f'rank{i+1}_caption' for i in range(10)] + [f'rank{i+1}_score' for i in range(10)]

with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for img_path, ranked in clip_results.items():
        row = {
            'filename':     img_path.name,
            'best_caption': ranked[0][0],
            'best_score':   round(float(ranked[0][1]), 4),
        }
        for i, (cap, score) in enumerate(ranked):
            row[f'rank{i+1}_caption'] = cap
            row[f'rank{i+1}_score']   = round(float(score), 4)
        writer.writerow(row)

print(f'Summary saved to: {OUTPUT_CSV}')
print(f'Rows written: {len(clip_results)}')